# Step 4: Nsight Python

Throughout this lab, we've been manually running `ncu` from the command line to profile kernels, then opening reports in the UI to analyze them. While this workflow is powerful, it can be tedious when you want to:

- Compare multiple kernel variants systematically
- Collect and visualize metrics programmatically
- Automate performance regression testing in CI/CD
- Generate publication-quality charts without manual export

[**Nsight Python**](https://docs.nvidia.com/nsight-python) is a new Python interface that automates performance analysis using NVIDIA Nsight Tools. With just a decorator, you can benchmark kernel variants, collect detailed architectural metrics, prevent GPU throttling, and generate visualizations — all from a single script. One of Nsight Python's strengths is automatic benchmarking of parameter spaces and visualization of its results. We will see in the example later on how to do this.

It's available on [GitHub](https://github.com/NVIDIA/nsight-python) and PyPI:
```
pip install -q nsight-python
```

---

## Collection

The core of Nsight Python are the `nsight.analyze.kernel` and `nsight.annotate` decorators/context managers. When you decorate a function and annotate the kernel in it, Nsight Python will:

1. **Profile** all GPU kernels launched within that function
2. **Collect** detailed metrics (configurable)
3. **Store** results for later comparison and visualization

The decorator is **non-invasive** — when profiling is disabled (the default), the decorated function runs normally with zero overhead. Profiling is enabled via a context manager. Here's the basic usage pattern:

```python
import nsight

@nsight.analyze.kernel
def my_gpu_workload():
    # Launch GPU kernels here
    # Nsight Python automatically profiles them
    with nsight.annotate('my-kernel'):
        kernel<<<grid, block>>>(args)
        torch.cuda.synchronize()
```

### nsight.analyze.kernel

A decorator that collects profiling data using NVIDIA Nsight Compute.
```python
class nsight.analyze.kernel(...)
```

Detailed signature:
```python
class nsight.analyze.kernel(
    _func: Callable[[...], None] | None = None,
    *,
    configs: Iterable[Any] | None = None,
    runs: int = 1,
    derive_metric: Callable[[...], float] | None = None,
    normalize_against: str | None = None,
    output: Literal['quiet', 'progress', 'verbose'] = 'progress',
    metrics: Sequence[str] = ['gpu__time_duration.sum'],
    ignore_kernel_list: Sequence[str] | None = None,
    clock_control: Literal['base', 'none'] = 'none',
    cache_control: Literal['all', 'none'] = 'all',
    replay_mode: Literal['kernel', 'range'] = 'kernel',
    thermal_control: bool = True,
    combine_kernel_metrics: Callable[[float, float], float] | None = None,
    output_prefix: str | None = None,
    output_csv: bool = False,
    )
```


The decorator returns a wrapped version of your function with the following signature:
```python
def wrapped_function(*args, configs=None, **kwargs) -> ProfileResults
```

Example:

```python
@nsight.analyze.kernel(metrics=[
    'sm__throughput.avg.pct_of_peak_sustained_elapsed',
    'gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed',
    'gpu__time_duration.sum'])
def run_attention_baseline():
...
```

### nsight.annotate

A decorator/context-manager hybrid for marking profiling regions. The encapsulated code will be profiled and associated with an NVTX range of the given annotate name.
```python
class nsight.annotate(name: str, ignore_failures: bool = False)
```

Example:

```python
def run_attention_baseline():
    with nsight.annotate("attention"):
        attention_prefill_dot[(B*H, SQ)](...)
        torch.cuda.synchronize()
```

---

### Accessing the Results

`@nsight.analyze.kernel` returns a `ProfileResults` object, which can be converted to a [dataframe](https://docs.nvidia.com/nsight-python/collection/core.html#nsight.collection.core.ProfileResults) for further usage:

```python 
results = decorated_function()
df = results.to_dataframe()
print(df[["Annotation", "Metric", "AvgValue"]].to_string(index=False))
```

This DataFrame contains aggregated statistics across multiple runs for each configuration and annotation combination. The data is equivalent to what is written to the processed_data-<function_name>-<run_id>.csv file when `output_csv=True`.

- **Annotation**: Name of the annotated region being profiled
- **AvgValue**: Average metric values across all runs
- **StdDev**: Standard deviation of the metrics across runs
- **MinValue**: Minimum metric values observed
- **MaxValue**: Maximum metric values observed
- **NumRuns**: Number of runs used for aggregation
- **Metric**: The metrics being collected
- **Kernel**: Name of the GPU kernel(s) launched
- ...


---

## Exercise: Profile the Attention Kernel with Nsight Python

<div class="alert alert-block alert-info">
    <p><b>Exercise</b>: Add the <code>@nsight.analyze.kernel</code> decorator and <code>nsight.annotate</code> context manager to profile our kernel</p>
</div>

Below is the `attention_prefill_dot` kernel from Step 3. Your task is to:

1. **Review the code** — check where the `@nsight.analyze.kernel` decorator and <code>nsight.annotate</code> context manager should be added.
2. **Make Updates** - Have it collect the following `metrics=['sm__throughput.avg.pct_of_peak_sustained_elapsed','gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed','gpu__time_duration.sum']` and print the results
3. **Run the first cell** to write the script to a file
4. **Run the second cell** to execute the script and see the profiling results

Hint: The decorator should go on the *host function* that launches the kernel, not on the Triton kernel itself. The context manager wraps the *kernel launch*.

<div class="alert alert-block alert-warning">
    Nsight Python 0.9 works by launching a subprocess with ncu, so it must be run as a standalone Python script (not inside a Jupyter notebook). In this step, we'll write our kernel to a file and run it from the command line. Future versions won't have this limitation anymore.
</div>

In [ ]:
%%writefile /dli/task/ncu/source/nsight_python_demo.py
import torch
import triton
import triton.language as tl
import nsight

@triton.jit
def attention_prefill_dot(
    Q_ptr, K_ptr, V_ptr, Out_ptr,
    stride_qb, stride_qm, stride_qk,
    stride_kb, stride_kn, stride_kk,
    stride_vb, stride_vn, stride_vk,
    stride_ob, stride_om, stride_ok,
    seq_q, seq_kv, scale,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, HEAD_DIM: tl.constexpr,
):
    """Baseline: One query per block. No K/V reuse across queries."""
    batch_head = tl.program_id(0)
    query_idx = tl.program_id(1)

    if query_idx >= seq_q:
        return

    q_ptrs = Q_ptr + batch_head * stride_qb + query_idx * stride_qm + tl.arange(0, HEAD_DIM)
    q = tl.load(q_ptrs)

    m_i = float('-inf')
    l_i = 0.0
    acc = tl.zeros([HEAD_DIM], dtype=tl.float32)

    for n_start in range(0, seq_kv, BLOCK_N):
        n_offs = n_start + tl.arange(0, BLOCK_N)
        n_mask = n_offs < seq_kv

        k_ptrs = K_ptr + batch_head * stride_kb + n_offs[:, None] * stride_kn + tl.arange(0, HEAD_DIM)[None, :]
        k = tl.load(k_ptrs, mask=n_mask[:, None], other=0.0)

        k_t = tl.trans(k)
        qk_2d = tl.dot(q[None, :], k_t.to(tl.float16))
        qk = tl.sum(qk_2d, axis=0) * scale

        m_new = tl.maximum(m_i, tl.max(qk))
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(qk - m_new)
        p = tl.where(n_mask, p, 0.0)
        l_new = alpha * l_i + tl.sum(p)

        v_ptrs = V_ptr + batch_head * stride_vb + n_offs[:, None] * stride_vn + tl.arange(0, HEAD_DIM)[None, :]
        v = tl.load(v_ptrs, mask=n_mask[:, None], other=0.0)
        pv_2d = tl.dot(p[None, :].to(tl.float16), v)
        acc = alpha * acc + tl.sum(pv_2d, axis=0)
        m_i, l_i = m_new, l_new

    acc = acc / l_i
    o_ptrs = Out_ptr + batch_head * stride_ob + query_idx * stride_om + tl.arange(0, HEAD_DIM)
    tl.store(o_ptrs, acc.to(tl.float16))


def run_baseline():
    """Host function that launches the attention kernel."""
    # Setup tensors
    torch.manual_seed(42)
    B, H, SQ, SKV, D = 8, 32, 64, 4096, 64
    q = torch.randn(B*H, SQ, D, dtype=torch.float16, device='cuda')
    k = torch.randn(B*H, SKV, D, dtype=torch.float16, device='cuda')
    v = torch.randn(B*H, SKV, D, dtype=torch.float16, device='cuda')
    out = torch.empty_like(q)

    # Launch the baseline attention kernel
    attention_prefill_dot[(B*H, SQ)](
        q, k, v, out,
        *q.stride(), *k.stride(), *v.stride(), *out.stride(),
        SQ, SKV, 1.0/(D**0.5),
        BLOCK_M=1, BLOCK_N=64, HEAD_DIM=D
    )
    torch.cuda.synchronize()


if __name__ == "__main__":
    session = run_baseline()
    df = session.to_dataframe()
    print(df[["Annotation", "Metric", "AvgValue"]].to_string(index=False))


Run the script from the command line. Nsight Python will automatically invoke `ncu` and collect metrics:

In [ ]:
!python3 /dli/task/ncu/source/nsight_python_demo.py

---

<details>
<summary><b>Click to reveal solution</b></summary>

Add the `@nsight.analyze.kernel` decorator to the `run_baseline()` function and `nsight.annotate` around the kernel launch.
You can copy the metrics list from the example on the top:

```python
@nsight.analyze.kernel(metrics=[
    'sm__throughput.avg.pct_of_peak_sustained_elapsed',
    'gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed',
    'gpu__time_duration.sum'
])
def run_baseline():
    """Host function that launches the attention kernel."""
    ... 
    with nsight.annotate("attention"):
        attention_prefill_dot(...)
        torch.cuda.synchronize()
    ...
```

The decorator goes on the **host function** (`run_baseline`), not on the Triton kernel. This is because:
- The Triton kernel is compiled to GPU code — it can't be instrumented by Python decorators  
- The host function is where Nsight Python can intercept and profile the kernel launch

</details>

Run the cell below to execute the version with the decorator already added:

In [ ]:
%%writefile /dli/task/ncu/source/nsight_python_solution.py
import torch
import triton
import triton.language as tl
import nsight

@triton.jit
def attention_prefill_dot(
    Q_ptr, K_ptr, V_ptr, Out_ptr,
    stride_qb, stride_qm, stride_qk,
    stride_kb, stride_kn, stride_kk,
    stride_vb, stride_vn, stride_vk,
    stride_ob, stride_om, stride_ok,
    seq_q, seq_kv, scale,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, HEAD_DIM: tl.constexpr,
):
    """Baseline: One query per block. No K/V reuse across queries."""
    batch_head = tl.program_id(0)
    query_idx = tl.program_id(1)

    if query_idx >= seq_q:
        return

    q_ptrs = Q_ptr + batch_head * stride_qb + query_idx * stride_qm + tl.arange(0, HEAD_DIM)
    q = tl.load(q_ptrs)

    m_i = float('-inf')
    l_i = 0.0
    acc = tl.zeros([HEAD_DIM], dtype=tl.float32)

    for n_start in range(0, seq_kv, BLOCK_N):
        n_offs = n_start + tl.arange(0, BLOCK_N)
        n_mask = n_offs < seq_kv

        k_ptrs = K_ptr + batch_head * stride_kb + n_offs[:, None] * stride_kn + tl.arange(0, HEAD_DIM)[None, :]
        k = tl.load(k_ptrs, mask=n_mask[:, None], other=0.0)

        k_t = tl.trans(k)
        qk_2d = tl.dot(q[None, :], k_t.to(tl.float16))
        qk = tl.sum(qk_2d, axis=0) * scale

        m_new = tl.maximum(m_i, tl.max(qk))
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(qk - m_new)
        p = tl.where(n_mask, p, 0.0)
        l_new = alpha * l_i + tl.sum(p)

        v_ptrs = V_ptr + batch_head * stride_vb + n_offs[:, None] * stride_vn + tl.arange(0, HEAD_DIM)[None, :]
        v = tl.load(v_ptrs, mask=n_mask[:, None], other=0.0)
        pv_2d = tl.dot(p[None, :].to(tl.float16), v)
        acc = alpha * acc + tl.sum(pv_2d, axis=0)
        m_i, l_i = m_new, l_new

    acc = acc / l_i
    o_ptrs = Out_ptr + batch_head * stride_ob + query_idx * stride_om + tl.arange(0, HEAD_DIM)
    tl.store(o_ptrs, acc.to(tl.float16))


# SOLUTION: @nsight.analyze.kernel decorator added!
@nsight.analyze.kernel(metrics=[
    'sm__throughput.avg.pct_of_peak_sustained_elapsed',
    'gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed',
    'gpu__time_duration.sum'
])
def run_baseline():
    """Host function that launches the attention kernel."""
    # Setup tensors
    torch.manual_seed(42)
    B, H, SQ, SKV, D = 8, 32, 64, 4096, 64
    q = torch.randn(B*H, SQ, D, dtype=torch.float16, device='cuda')
    k = torch.randn(B*H, SKV, D, dtype=torch.float16, device='cuda')
    v = torch.randn(B*H, SKV, D, dtype=torch.float16, device='cuda')
    out = torch.empty_like(q)

    # Launch the baseline attention kernel
    # SOLUTION: context manager added
    with nsight.annotate("attention"):
        attention_prefill_dot[(B*H, SQ)](
            q, k, v, out,
            *q.stride(), *k.stride(), *v.stride(), *out.stride(),
            SQ, SKV, 1.0/(D**0.5),
            BLOCK_M=1, BLOCK_N=64, HEAD_DIM=D
        )
        torch.cuda.synchronize()


if __name__ == "__main__":
    session = run_baseline()
    df = session.to_dataframe()
    print(df[["Annotation", "Metric", "AvgValue"]].to_string(index=False))

In [ ]:
!python3 /dli/task/ncu/source/nsight_python_solution.py

---

### Expected Results

When running the modified script, Nsight Python will:

1. Automatically launch `ncu` to profile the kernel
2. Collect detailed metrics
3. Display a summary showing key performance indicators

The output will show metrics similar to what we observed when profiling manually in Step 3:
- **Duration**: ~5 ms
- **Memory Throughput**: ~85% (memory-bound kernel)
- **SM Throughput**: ~50%

This confirms that Nsight Python produces the same insights as the manual `ncu` workflow — but with far less effort!

---

## Summary

<div class="alert alert-block alert-success">
    <p><b>Nsight Python</b> enables in-script GPU kernel profiling with simple decorators.</p>
    <p><b><code>@nsight.analyze.kernel</code></b>: Decorate host functions that launch GPU kernels. Returns results.</p>
    <p><b><code>nsight.annotate</code></b>: Context manager for wrapping kernels in decorated functions.</p>
    <p>Both are needed for successful profiling</p>
    <p><b>Standalone scripts</b>: Nsight Python must run as a script (not inside Jupyter) because it launches <code>ncu</code> as a subprocess.</p>
</div>

**Congratulations!** You have completed the hands-on exercises. Continue to the **[Lab Summary](summary.ipynb)** for key learnings and further resources.